### Первое задание в Azure - было съедено волками.

In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, f1_score

In [2]:

# 1. Загрузка данных и взятие первых 600 строк
df = pd.read_csv('diabetes.csv')
df_600 = df.iloc[:600].copy()

# Подсчет количества строк класса 0
count_class_0 = (df_600['Outcome'] == 0).sum()
print(f"1. Количество строк класса 0 в первых 600: {count_class_0}")

# 2. Разделение на train/test (80/20)
# Первые 80% строк — train, остальные 20% — test
split_idx = int(len(df_600) * 0.8)
train_df = df_600.iloc[:split_idx]
test_df = df_600.iloc[split_idx:]

features = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 
            'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
target = 'Outcome'

X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

# 3. Обучение DecisionTreeClassifier
clf = DecisionTreeClassifier(
    criterion='entropy',
    max_leaf_nodes=5,
    min_samples_leaf=5,
    random_state=2020
)
clf.fit(X_train, y_train)

# 4. Изучение дерева
print(f"\n4. Глубина дерева: {clf.get_depth()}")

# Вывод текстового представления дерева, чтобы найти последний уровень
tree_rules = export_text(clf, feature_names=features)
print("\nТекстовое представление дерева:")
print(tree_rules)

# Поиск предиктора на последнем уровне (вручную по выводу или через код)
# В sklearn tree_.feature содержит индексы признаков для каждого узла (-2 для листьев)
# Найдем признаки, которые используются в листьях (или на максимальной глубине)
tree = clf.tree_
last_level_features = set()
for i in range(tree.node_count):
    if tree.children_left[i] == -1: # Это лист
        # Ищем его родителя, чтобы понять, по какому признаку было разделение
        # (В sklearn нет прямого доступа к родителю, поэтому проще найти по глубине)
        pass

# Альтернативный способ: вывести все разделения и их глубину
def print_splits(node_id=0, depth=0):
    if clf.tree_.children_left[node_id] == -1:
        return
    feature_name = features[clf.tree_.feature[node_id]]
    threshold = clf.tree_.threshold[node_id]
    print(f"Глубина {depth}: {feature_name} <= {threshold:.2f}")
    print_splits(clf.tree_.children_left[node_id], depth + 1)
    print_splits(clf.tree_.children_right[node_id], depth + 1)

print("\nВсе разделения в дереве:")
print_splits()

# 5. Оценка модели
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')

print(f"\n5. Доля правильных ответов (Accuracy): {accuracy:.2f}")
print(f"Среднее значение метрик F1 (Macro-F1): {macro_f1:.2f}")

# 6. Предсказание для конкретных пациентов из начального набора
# Индексы: 746, 725, 719, 721 (нумерация с нуля)
indices = [746, 725, 719, 721]
patients = df.iloc[indices][features]
predictions = clf.predict(patients)

print("\n6. Предсказания для пациентов:")
for idx, pred in zip(indices, predictions):
    print(f"Пациент {idx}: {pred}")

1. Количество строк класса 0 в первых 600: 392

4. Глубина дерева: 4

Текстовое представление дерева:
|--- Glucose <= 154.50
|   |--- BMI <= 26.30
|   |   |--- class: 0
|   |--- BMI >  26.30
|   |   |--- Glucose <= 99.50
|   |   |   |--- class: 0
|   |   |--- Glucose >  99.50
|   |   |   |--- Age <= 28.50
|   |   |   |   |--- class: 0
|   |   |   |--- Age >  28.50
|   |   |   |   |--- class: 1
|--- Glucose >  154.50
|   |--- class: 1


Все разделения в дереве:
Глубина 0: Glucose <= 154.50
Глубина 1: BMI <= 26.30
Глубина 2: Glucose <= 99.50
Глубина 3: Age <= 28.50

5. Доля правильных ответов (Accuracy): 0.79
Среднее значение метрик F1 (Macro-F1): 0.75

6. Предсказания для пациентов:
Пациент 746: 0
Пациент 725: 1
Пациент 719: 0
Пациент 721: 0


## Часть с Azure теперь не у волков, а у медведей. Они ещё страшнее

In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans

# 1. Загрузка данных
df = pd.read_csv('tipa.csv')
# Убираем пробелы в названиях колонок, если они есть
df.columns = df.columns.str.strip()

# Извлекаем координаты X и Y
X_data = df[['X', 'Y']].values

# 2. Параметры инициализации из задания
init_centroids = np.array([
    [12.67, 12.0],  # Кластер 0
    [8.0, 8.5],     # Кластер 1
    [5.57, 14.71]   # Кластер 2
])

# 3. Обучение KMeans
kmeans = KMeans(
    n_clusters=3, 
    init=init_centroids, 
    max_iter=100, 
    n_init=1, 
    random_state=42  # Фиксируем для воспроизводимости (хотя при заданном init это не обязательно)
)
kmeans.fit(X_data)

# 4. Получаем метки кластеров и финальные центроиды
labels = kmeans.labels_
centroids = kmeans.cluster_centers_

print("Финальные центроиды:")
print(centroids)

# 5. Выбираем объекты, попавшие в кластер 0
cluster_0_points = X_data[labels == 0]
cluster_0_centroid = centroids[0] # Центроид кластера 0

print(f"\nКоличество объектов в кластере 0: {len(cluster_0_points)}")
print(f"Центроид кластера 0: {cluster_0_centroid}")

# 6. Считаем евклидово расстояние от каждого объекта кластера 0 до его центроида
distances = np.linalg.norm(cluster_0_points - cluster_0_centroid, axis=1)

print("\nРасстояния объектов кластера 0 до центроида:")
for i, dist in enumerate(distances):
    print(f"Точка {cluster_0_points[i]}: {dist:.4f}")

# 7. Среднее расстояние
mean_distance = np.mean(distances)
print(f"\nСреднее расстояние: {mean_distance:.3f}")

Финальные центроиды:
[[12.4        17.        ]
 [10.16666667  5.5       ]
 [ 2.         18.5       ]]

Количество объектов в кластере 0: 5
Центроид кластера 0: [12.4 17. ]

Расстояния объектов кластера 0 до центроида:
Точка [11 19]: 2.4413
Точка [11 16]: 1.7205
Точка [11 17]: 1.4000
Точка [12 13]: 4.0200
Точка [17 20]: 5.4918

Среднее расстояние: 3.015
